# Interactive Event Display & Debugging Dashboard

This notebook provides an interactive dashboard to visualize 3D events (truth tracks and simulated hits) and 2D projections using **Panel** and **Plotly**.

## Features

- **Interactive Event Selection**: Browse through events in your dataset
- **Truth Track Visualization**: View the original track segments from edep-sim
- **Integrated Simulation**: Run the LAr-ND simulation directly on events to generate simulated hits
- **Multi-View Display**: 
  - 3D visualization showing truth tracks and simulated hits together
  - 2D projections (X-Y, Z-Y) for detailed comparison
  - Energy/ADC distribution histograms
- **Flexible Display Options**: Toggle visibility of truth tracks, original hits, and simulated hits

## Setup

Ensure you have the required libraries installed. If not, uncomment and run the cell below.

In [28]:
!pip install panel plotly pandas anywidget

/usr/lib/python3.10/pty.py:89: RuntimeWarning:

os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.



Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 864.3/864.3 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [anywidget]/2 [anywidget]

[notice] A new release of pip is available: 25.1.1 -> 26.0
[notice] To update, run: python3 -m pip install --upgrade pip


In [1]:
import sys, os
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# Add project root to path
project_root = '/sdf/group/neutrino/pgranger/larnd-sim-jax'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from larndsim.consts_jax import build_params_class, load_detector_properties, load_lut
from larndsim.losses_jax import llhd_loss, adc2charge, mse_adc
from optimize.strategies import LUTSimulation, LUTProbabilisticSimulation, ProbabilisticLossStrategy, CollapsedProbabilisticLossStrategy, GenericLossStrategy
from optimize.dataio import TracksDataset, DataLoader
from optimize.ranges import ranges

%cd /sdf/group/neutrino/pgranger/larnd-sim-jax

/sdf/group/neutrino/pgranger/larnd-sim-jax


/sdf/home/p/pgranger/.local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [4]:
# --- Load Data ---

# --- Configuration ---

INPUT_FILE = '/sdf/data/neutrino/cyifan/diffsim_input/true_proton_edep_2cm.h5'
LUT_FILE = 'src/larndsim/detector_properties/response_44.npy'
DET_PROPS = 'src/larndsim/detector_properties/module0.yaml'
PIXEL_LAYOUTS = 'src/larndsim/pixel_layouts/multi_tile_layout-2.4.16_v4.yaml'

# Parameters to optimize (and their ranges)

RELEVANT_PARAMS = ['Ab', 'kb', 'lifetime', 'tran_diff', 'long_diff', 'eField', 'shift_x', 'shift_y', 'shift_z']

# Simulation Settings
ELECTRON_SAMPLING_RESOLUTION = 0.01
NUMBER_PIX_NEIGHBORS = 4
SIGNAL_LENGTH = 250

dataset = TracksDataset(filename=INPUT_FILE, ntrack=-1, max_nbatch=1, print_input=False, max_batch_len=200,
                        electron_sampling_resolution=ELECTRON_SAMPLING_RESOLUTION)
dataloader = DataLoader(dataset, batch_size=100)

# Get one batch
tracks_batch = dataloader[0]
track_fields = dataset.get_track_fields()

# Prepare tracks for JAX
tracks = jax.device_put(tracks_batch.reshape(-1, len(track_fields)))

print(f"Loaded {tracks.shape[0]} tracks segments.")

: 

: 

: 

In [2]:
!ls -lht '/sdf/data/neutrino/cyifan/diffsim_input/true_proton_edep_2cm.h5'

-rw-r--r-- 1 18025 nu 1.4G Aug 15 19:11 /sdf/data/neutrino/cyifan/diffsim_input/true_proton_edep_2cm.h5


In [3]:
# --- Setup Parameters ---

ParamsClass = build_params_class(RELEVANT_PARAMS)
ref_params = load_detector_properties(ParamsClass, DET_PROPS, PIXEL_LAYOUTS)
ref_params = ref_params.replace(
    electron_sampling_resolution=ELECTRON_SAMPLING_RESOLUTION,
    number_pix_neighbors=NUMBER_PIX_NEIGHBORS,
    signal_length=SIGNAL_LENGTH,
    time_window=SIGNAL_LENGTH,
    # Ensure noise params are zero for deterministic checks if needed, but llhd handles noise in prob
)

response, ref_params = load_lut(LUT_FILE, ref_params)

# Set nominal values
initial_values = {p: ranges[p]['nom'] for p in RELEVANT_PARAMS}
current_params = ref_params.replace(**initial_values)

print("Parameters initialized.")

INFO:larndsim.consts_jax:Loading response from numpy array
Could not load symbol cuFuncGetName. Error: /.singularity.d/libs/libcuda.so.1: undefined symbol: cuFuncGetName


Parameters initialized.


## Testing & Debugging

Run the cells below to test the simulation directly and ensure everything is working:

### Alternative: Direct Display Without Panel Server

If the Panel server isn't working well in your environment, you can display the plots directly:

In [4]:
target_strategy = LUTSimulation(response)

rngkey = 42

# Run stochastic simulation to get 'fake data'

target_output = target_strategy.predict(current_params, tracks, track_fields, rngkey)

In [5]:
target_output.keys()

dict_keys(['adcs', 'pixel_x', 'pixel_y', 'pixel_z', 'ticks', 'hit_prob', 'event', 'hit_pixels', 'unique_pixels', 'wfs'])

In [6]:
# Check the structure of target_output
print("Keys in target_output:")
for key in target_output.keys():
    val = target_output[key]
    if hasattr(val, 'shape'):
        print(f"  {key}: shape = {val.shape}, dtype = {val.dtype}")
    else:
        print(f"  {key}: {type(val)}")

Keys in target_output:
  adcs: shape = (1859,), dtype = float32
  pixel_x: shape = (1859,), dtype = float32
  pixel_y: shape = (1859,), dtype = float32
  pixel_z: shape = (1859,), dtype = float32
  ticks: shape = (1859,), dtype = float32
  hit_prob: shape = (1859,), dtype = float32
  event: shape = (1859,), dtype = int32
  hit_pixels: shape = (1859,), dtype = int32
  unique_pixels: shape = (576,), dtype = int32
  wfs: shape = (576, 2000), dtype = float32


In [7]:
# Check the mapping for a specific hit
hit_idx = 100
pixel_id = target_output['hit_pixels'][hit_idx]
print(f"Hit {hit_idx} -> pixel_id = {pixel_id}")

# Find which waveform index this corresponds to
wf_idx = np.where(target_output['unique_pixels'] == pixel_id)[0]
print(f"pixel_id {pixel_id} -> waveform index = {wf_idx}")

if len(wf_idx) > 0:
    wf_idx = wf_idx[0]
    waveform = target_output['wfs'][wf_idx]
    print(f"Waveform shape: {waveform.shape}")
    print(f"Waveform range: [{waveform.min():.3f}, {waveform.max():.3f}]")

Hit 100 -> pixel_id = 14678
pixel_id 14678 -> waveform index = [79]
Waveform shape: (2000,)
Waveform range: [-4865.537, 13615.047]


In [8]:
# Check which hits belong to this pixel
hits_on_pixel = np.where(target_output['hit_pixels'] == pixel_id)[0]
print(f"\nPixel {pixel_id} has {len(hits_on_pixel)} hits:")
for h_idx in hits_on_pixel[:5]:  # Show first 5
    tick = target_output['ticks'][h_idx]
    adc = target_output['adcs'][h_idx]
    time_us = tick * float(current_params.t_sampling)
    print(f"  Hit {h_idx}: tick={tick:.1f}, time={time_us:.2f} µs, ADC={adc:.1f}")


Pixel 14678 has 3 hits:
  Hit 98: tick=1063.0, time=106.30 µs, ADC=82.5
  Hit 99: tick=1105.0, time=110.50 µs, ADC=93.1
  Hit 100: tick=1139.0, time=113.90 µs, ADC=81.9


In [14]:
# --- 3D Visualization: Truth Segments + Target Output ---

import plotly.graph_objects as go

# Extract target output data
target_adcs = np.array(target_output['adcs'])
target_x = np.array(target_output['pixel_x'])
target_y = np.array(target_output['pixel_y'])
target_z = np.array(target_output['pixel_z'])
target_hit_prob = np.array(target_output['hit_prob'])

print(f"Target output shapes: adcs={target_adcs.shape}, x={target_x.shape}, z={target_z.shape}")

# Flatten if needed (handle 2D output from stochastic)
if target_adcs.ndim == 2:
    n_pix, n_hits = target_adcs.shape
    print(f"Flattening: {n_pix} pixels × {n_hits} hits")
    target_adcs = target_adcs.flatten()
    target_z = target_z.flatten()
    target_hit_prob = target_hit_prob.flatten()
    target_x = np.repeat(target_x, n_hits)
    target_y = np.repeat(target_y, n_hits)

# Filter valid hits
mask = (target_hit_prob > 0) & (target_adcs > 0)
print(f"Valid hits: {mask.sum()} / {len(mask)}")

# Create figure
fig = go.Figure()

# Add truth segments as lines (vectorized for speed)
tracks_np = np.array(tracks)  # Convert from JAX to numpy once
x_start_idx = track_fields.index('x_start')
x_end_idx = track_fields.index('x_end')
y_start_idx = track_fields.index('y_start')
y_end_idx = track_fields.index('y_end')
z_start_idx = track_fields.index('z_start')
z_end_idx = track_fields.index('z_end')

# Vectorized construction of line segments
n_tracks = len(tracks_np)
x_lines = np.empty(n_tracks * 3)
y_lines = np.empty(n_tracks * 3)
z_lines = np.empty(n_tracks * 3)

x_lines[0::3] = tracks_np[:, x_start_idx]
x_lines[1::3] = tracks_np[:, x_end_idx]
x_lines[2::3] = np.nan

y_lines[0::3] = tracks_np[:, y_start_idx]
y_lines[1::3] = tracks_np[:, y_end_idx]
y_lines[2::3] = np.nan

z_lines[0::3] = tracks_np[:, z_start_idx]
z_lines[1::3] = tracks_np[:, z_end_idx]
z_lines[2::3] = np.nan

fig.add_trace(go.Scatter3d(
    x=x_lines, 
    y=y_lines, 
    z=z_lines,
    mode='lines',
    name='Truth Segments',
    line=dict(color='blue', width=4)
))

# Add target output hits as 3D voxels instead of points
# Calculate voxel dimensions
pixel_pitch = float(current_params.pixel_pitch)  # pixel size in cm
interval = round((3 * current_params.CLOCK_CYCLE + current_params.ADC_HOLD_DELAY * current_params.CLOCK_CYCLE) / current_params.t_sampling)
time_interval_cm = interval * float(current_params.t_sampling) * float(current_params.vdrift)  # Convert time samples to cm in drift direction

print(f"\nVoxel dimensions:")
print(f"  Pixel pitch: {pixel_pitch} cm")
print(f"  Time interval: {interval} samples = {time_interval_cm:.3f} cm in drift")

# Function to create a box (voxel) mesh
def create_voxel(x_center, y_center, z_center, dx, dy, dz):
    """Create a 3D box mesh centered at (x,y,z) with dimensions (dx, dy, dz)"""
    # Define the 8 vertices of the box
    vertices = np.array([
        [x_center - dx/2, y_center - dy/2, z_center - dz/2],  # 0: bottom-left-front
        [x_center + dx/2, y_center - dy/2, z_center - dz/2],  # 1: bottom-right-front
        [x_center + dx/2, y_center + dy/2, z_center - dz/2],  # 2: bottom-right-back
        [x_center - dx/2, y_center + dy/2, z_center - dz/2],  # 3: bottom-left-back
        [x_center - dx/2, y_center - dy/2, z_center + dz/2],  # 4: top-left-front
        [x_center + dx/2, y_center - dy/2, z_center + dz/2],  # 5: top-right-front
        [x_center + dx/2, y_center + dy/2, z_center + dz/2],  # 6: top-right-back
        [x_center - dx/2, y_center + dy/2, z_center + dz/2],  # 7: top-left-back
    ])
    
    # Define the 12 triangular faces (2 per box face)
    faces = np.array([
        # Bottom face (z-)
        [0, 1, 2], [0, 2, 3],
        # Top face (z+)
        [4, 5, 6], [4, 6, 7],
        # Front face (y-)
        [0, 1, 5], [0, 5, 4],
        # Back face (y+)
        [2, 3, 7], [2, 7, 6],
        # Left face (x-)
        [0, 3, 7], [0, 7, 4],
        # Right face (x+)
        [1, 2, 6], [1, 6, 5],
    ])
    
    return vertices, faces

# Create voxels for each hit
masked_indices = np.where(mask)[0]
n_voxels = len(masked_indices)

# Limit number of voxels for performance (can be adjusted)
max_voxels = 5000
if n_voxels > max_voxels:
    print(f"  Limiting to {max_voxels} voxels (from {n_voxels}) for performance")
    # Sample evenly or by ADC value
    step = n_voxels // max_voxels
    display_indices = masked_indices[::step][:max_voxels]
else:
    display_indices = masked_indices

# Normalize ADC values for color mapping
adc_values = target_adcs[display_indices]
adc_min, adc_max = adc_values.min(), adc_values.max()
adc_normalized = (adc_values - adc_min) / (adc_max - adc_min + 1e-9)

# Use Plasma colorscale
import matplotlib.cm as cm
plasma = cm.get_cmap('plasma')

# Add each voxel as a Mesh3d trace
for idx, i in enumerate(display_indices):
    vertices, faces = create_voxel(
        target_x[i], target_y[i], target_z[i],
        pixel_pitch, pixel_pitch, time_interval_cm
    )
    
    # Map ADC to color
    rgba = plasma(adc_normalized[idx])
    color_str = f'rgba({int(rgba[0]*255)},{int(rgba[1]*255)},{int(rgba[2]*255)},{rgba[3]*0.6})'
    
    fig.add_trace(go.Mesh3d(
        x=vertices[:, 0],
        y=vertices[:, 1],
        z=vertices[:, 2],
        i=faces[:, 0],
        j=faces[:, 1],
        k=faces[:, 2],
        color=color_str,
        opacity=0.6,
        showlegend=(idx == 0),  # Only show legend for first voxel
        name=f'Target Voxels ({len(display_indices)} shown)',
        hovertemplate=f'<b>Hit {i}</b><br>ADC: {target_adcs[i]:.1f}<br>X: {target_x[i]:.2f}<br>Y: {target_y[i]:.2f}<br>Z: {target_z[i]:.2f}<extra></extra>'
    ))

print(f"  Created {len(display_indices)} voxels")

# Add detector geometry
# Get TPC boundaries from parameters (convert to numpy scalars)
tpc_borders = np.array(current_params.tpc_borders)
drift_length = float(current_params.drift_length)

print(f"\nDetector geometry:")
print(f"  TPC borders shape: {tpc_borders.shape}")
print(f"  Drift length: {drift_length}")

# TPC borders is 3D array: (n_tpcs, 3, 2) - multiple TPCs, each with [x, y, z] × [min, max]
# Get overall bounds across all TPCs
if tpc_borders.ndim == 3:
    # Multiple TPCs - get overall min/max
    x_min = float(tpc_borders[:, 0, 0].min())  # Min of all x_min
    x_max = float(tpc_borders[:, 0, 1].max())  # Max of all x_max
    y_min = float(tpc_borders[:, 1, 0].min())
    y_max = float(tpc_borders[:, 1, 1].max())
    z_min = float(tpc_borders[:, 2, 0].min())
    z_max = float(tpc_borders[:, 2, 1].max())
    print(f"  Found {tpc_borders.shape[0]} TPCs, using overall bounds")
elif tpc_borders.ndim == 2:
    # Single TPC format: [[x_min, x_max], [y_min, y_max], [z_min, z_max]]
    x_min, x_max = float(tpc_borders[0, 0]), float(tpc_borders[0, 1])
    y_min, y_max = float(tpc_borders[1, 0]), float(tpc_borders[1, 1])
    z_min, z_max = float(tpc_borders[2, 0]), float(tpc_borders[2, 1])
else:
    raise ValueError(f"Unexpected tpc_borders shape: {tpc_borders.shape}")

# Draw TPC box edges
edges = [
    # Bottom rectangle
    ([x_min, x_max], [y_min, y_min], [z_min, z_min]),
    ([x_max, x_max], [y_min, y_max], [z_min, z_min]),
    ([x_max, x_min], [y_max, y_max], [z_min, z_min]),
    ([x_min, x_min], [y_max, y_min], [z_min, z_min]),
    # Top rectangle
    ([x_min, x_max], [y_min, y_min], [z_max, z_max]),
    ([x_max, x_max], [y_min, y_max], [z_max, z_max]),
    ([x_max, x_min], [y_max, y_max], [z_max, z_max]),
    ([x_min, x_min], [y_max, y_min], [z_max, z_max]),
    # Vertical edges
    ([x_min, x_min], [y_min, y_min], [z_min, z_max]),
    ([x_max, x_max], [y_min, y_min], [z_min, z_max]),
    ([x_max, x_max], [y_max, y_max], [z_min, z_max]),
    ([x_min, x_min], [y_max, y_max], [z_min, z_max]),
]

for edge_x, edge_y, edge_z in edges:
    fig.add_trace(go.Scatter3d(
        x=edge_x, y=edge_y, z=edge_z,
        mode='lines',
        line=dict(color='gray', width=2),
        showlegend=False,
        hoverinfo='skip'
    ))

# Draw cathode plane (typically at one end of drift direction)
# Cathode is usually at z_min
cathode_z = z_min

# Create cathode rectangle
cathode_x = [x_min, x_max, x_max, x_min, x_min]
cathode_y = [y_min, y_min, y_max, y_max, y_min]
cathode_z_plane = [cathode_z] * 5

fig.add_trace(go.Scatter3d(
    x=cathode_x, 
    y=cathode_y, 
    z=cathode_z_plane,
    mode='lines',
    line=dict(color='red', width=4),
    name='Cathode Plane',
    hoverinfo='name'
))

# Add anode plane (at opposite end)
anode_z = z_max
anode_x = [x_min, x_max, x_max, x_min, x_min]
anode_y = [y_min, y_min, y_max, y_max, y_min]
anode_z_plane = [anode_z] * 5

fig.add_trace(go.Scatter3d(
    x=anode_x, 
    y=anode_y, 
    z=anode_z_plane,
    mode='lines',
    line=dict(color='green', width=4, dash='dash'),
    name='Anode Plane',
    hoverinfo='name'
))

fig.update_layout(
    title="Truth Segments + Target Output (3D) with Detector Geometry",
    scene=dict(
        xaxis_title='X (cm)',
        yaxis_title='Y (cm)',
        zaxis_title='Z (cm)',
        aspectmode='data'
    ),
    width=1200,
    height=800
)

fig.show()

print(f"\n✅ Plotted {len(tracks)} segments and {mask.sum()} target hits")
print(f"   TPC volume: X=[{x_min:.1f}, {x_max:.1f}], Y=[{y_min:.1f}, {y_max:.1f}], Z=[{z_min:.1f}, {z_max:.1f}] cm")
print(f"   Cathode at Z={cathode_z:.1f} cm (red), Anode at Z={anode_z:.1f} cm (green)")

Target output shapes: adcs=(1859,), x=(1859,), z=(1859,)
Valid hits: 1859 / 1859

Voxel dimensions:
  Pixel pitch: 0.4434 cm
  Time interval: 18 samples = 0.286 cm in drift


/lscratch/pgranger/tmp/ipykernel_2686580/2404119024.py:131: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.



  Created 1859 voxels

Detector geometry:
  TPC borders shape: (2, 3, 2)
  Drift length: 30.431
  Found 2 TPCs, using overall bounds



✅ Plotted 20143 segments and 1859 target hits
   TPC volume: X=[-31.0, 31.0], Y=[-62.1, 62.1], Z=[-30.6, 0.2] cm
   Cathode at Z=-30.6 cm (red), Anode at Z=0.2 cm (green)


In [57]:
target_output['adcs'].shape

(1859,)

In [20]:
# --- Function to Display Waveform for a Specific Hit ---

def show_hit_waveform(hit_index):
    """
    Display information and waveform for a specific hit.
    
    Args:
        hit_index: Index of the hit in the flattened arrays
    """
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    
    # Get hit information
    if hit_index >= len(target_adcs) or hit_index < 0:
        print(f"❌ Invalid hit index {hit_index}. Valid range: 0 to {len(target_adcs)-1}")
        return
    
    hit_x = target_x[hit_index]
    hit_y = target_y[hit_index]
    hit_z = target_z[hit_index]
    hit_adc = target_adcs[hit_index]
    hit_prob = target_hit_prob[hit_index]
    
    # Check if hit is valid
    if hit_prob == 0 or hit_adc == 0:
        print(f"⚠️  Hit {hit_index} has zero probability or ADC (likely filtered out)")
    
    print(f"📊 Hit #{hit_index} Information:")
    print(f"   Position: X={hit_x:.2f}, Y={hit_y:.2f}, Z={hit_z:.2f} cm")
    print(f"   ADC Value: {hit_adc:.1f}")
    print(f"   Hit Probability: {hit_prob:.3f}")
    
    # Get the pixel ID this hit belongs to
    pixel_id = int(target_output['hit_pixels'][hit_index])
    print(f"   Pixel ID: {pixel_id}")
    
    # Find the waveform index for this pixel
    wf_indices = np.where(target_output['unique_pixels'] == pixel_id)[0]
    if len(wf_indices) == 0:
        print(f"❌ No waveform found for pixel ID {pixel_id}")
        return
    wf_idx = int(wf_indices[0])
    print(f"   Waveform Index: {wf_idx}")
    
    # Check if we have waveform data
    if 'wfs' in target_output:
        # Get the waveform for this pixel using the waveform index
        pixel_waveform = np.array(target_output['wfs'][wf_idx])
        time_ticks = np.arange(len(pixel_waveform)) * float(current_params.t_sampling)
        
        # Find all hits that belong to this pixel
        hits_on_pixel = np.where(target_output['hit_pixels'] == pixel_id)[0]
        n_hits_on_pixel = len(hits_on_pixel)
        print(f"   Number of hits on this pixel: {n_hits_on_pixel}")
        
        # Create figure with 2 subplots: waveform and voxel location
        fig = make_subplots(
            rows=1, cols=2,
            column_widths=[0.6, 0.4],
            subplot_titles=(
                f"Waveform for Hit #{hit_index} (Pixel ID {pixel_id}, {n_hits_on_pixel} hits)",
                f"Hit Location"
            ),
            specs=[[{"type": "scatter"}, {"type": "scatter3d"}]]
        )
        
        # Add waveform
        fig.add_trace(
            go.Scatter(
                x=time_ticks,
                y=pixel_waveform,
                mode='lines+markers',
                name='Current Signal',
                line=dict(color='orange', width=2),
                marker=dict(size=4)
            ),
            row=1, col=1
        )
        
        # Add vertical lines for all hits on this pixel
        if 'ticks' in target_output:
            for h_idx in hits_on_pixel:
                hit_tick = float(target_output['ticks'][h_idx])
                hit_time = hit_tick * float(current_params.t_sampling)
                hit_adc_val = float(target_output['adcs'][h_idx])
                
                # Highlight the current hit differently
                if h_idx == hit_index:
                    line_color = "red"
                    line_width = 3
                    opacity = 0.9
                else:
                    line_color = "pink"
                    line_width = 2
                    opacity = 0.5
                
                fig.add_vline(
                    x=hit_time,
                    line_dash="dash",
                    line_color=line_color,
                    line_width=line_width,
                    opacity=opacity,
                    row=1, col=1,
                    annotation_text=f"ADC:{hit_adc_val:.0f}",
                    annotation_position="top"
                )
            
            print(f"   Hit times (µs): {[float(target_output['ticks'][h]) * float(current_params.t_sampling) for h in hits_on_pixel]}")
        else:
            print(f"   Warning: 'ticks' not found in target_output")
        
        # Add truth segment arrival times
        # Find truth segments that are near this pixel in x,y and calculate their drift time
        pixel_tolerance = 2.0 * pixel_pitch  # Look within 2 pixel pitches
        vdrift = float(current_params.vdrift)  # cm/µs
        
        # Get detector geometry for drift calculation
        tpc_borders = np.array(current_params.tpc_borders)
        if tpc_borders.ndim == 3:
            z_min = float(tpc_borders[:, 2, 0].min())
            z_max = float(tpc_borders[:, 2, 1].max())
        else:
            z_min = float(tpc_borders[2, 0])
            z_max = float(tpc_borders[2, 1])
        
        # Determine which end is the anode (readout plane)
        # Typically cathode is at one end, anode at the other
        # We need to figure out which is which based on the actual z positions of hits
        # If hit z positions are closer to z_min, then cathode is at z_max
        # If hit z positions are closer to z_max, then cathode is at z_min
        
        # Simple heuristic: readout is where the hits actually are
        # Since hits are detected at the readout (anode), use the z value of hits
        mean_hit_z = np.mean([target_z[h] for h in hits_on_pixel])
        
        # Anode is whichever end is closer to the hit positions
        if abs(mean_hit_z - z_min) < abs(mean_hit_z - z_max):
            anode_z = z_min
            cathode_z = z_max
        else:
            anode_z = z_max
            cathode_z = z_min
        
        # Get the pixel x,y position (all hits on same pixel have same x,y)
        pixel_x = hit_x
        pixel_y = hit_y
        
        truth_arrival_times = []
        truth_energies = []
        truth_z_positions = []
        
        # Check if we have energy information in tracks
        if 'dE' in track_fields:
            dE_idx = track_fields.index('dE')
            has_energy = True
        else:
            has_energy = False
        
        for track in tracks_np:
            x_start = track[x_start_idx]
            y_start = track[y_start_idx]
            z_start = track[z_start_idx]
            x_end = track[x_end_idx]
            y_end = track[y_end_idx]
            z_end = track[z_end_idx]
            
            # Check if segment is near the pixel in x,y
            # Distance from point to line segment in x,y plane
            dx = x_end - x_start
            dy = y_end - y_start
            length_xy = np.sqrt(dx**2 + dy**2)
            
            if length_xy > 0:
                # Project pixel onto line segment
                t = max(0, min(1, ((pixel_x - x_start) * dx + (pixel_y - y_start) * dy) / (length_xy**2)))
                closest_x = x_start + t * dx
                closest_y = y_start + t * dy
                dist_xy = np.sqrt((pixel_x - closest_x)**2 + (pixel_y - closest_y)**2)
            else:
                # Point segment
                dist_xy = np.sqrt((pixel_x - x_start)**2 + (pixel_y - y_start)**2)
                closest_x = x_start
                closest_y = y_start
                t = 0
            
            # If segment is near the pixel, calculate arrival time
            if dist_xy <= pixel_tolerance:
                # Get z position at closest point
                closest_z = z_start + t * (z_end - z_start)
                
                # Calculate drift distance from ionization point to anode
                drift_distance = abs(anode_z - closest_z)
                
                # Calculate drift time
                arrival_time = drift_distance / vdrift  # µs
                
                truth_arrival_times.append(arrival_time)
                truth_z_positions.append(closest_z)
                
                if has_energy:
                    truth_energies.append(track[dE_idx])
                else:
                    truth_energies.append(1.0)
        
        # Add markers for truth segment arrival times
        if truth_arrival_times:
            print(f"   Found {len(truth_arrival_times)} truth segments near pixel")
            print(f"   Detector: Anode at Z={anode_z:.2f} cm, Cathode at Z={cathode_z:.2f} cm")
            print(f"   Drift velocity: {vdrift:.4f} cm/µs")
            print(f"   Truth z positions (cm): {[f'{z:.2f}' for z in truth_z_positions[:5]]} ...")
            print(f"   Drift distances (cm): {[f'{abs(anode_z - z):.2f}' for z in truth_z_positions[:5]]} ...")
            print(f"   Truth arrival times (µs): {[f'{t:.2f}' for t in truth_arrival_times[:10]]}")
            
            for arrival_time, energy in zip(truth_arrival_times, truth_energies):
                # Size marker by energy if available
                marker_size = 8 if not has_energy else min(20, 5 + energy * 100)
                
                fig.add_trace(
                    go.Scatter(
                        x=[arrival_time],
                        y=[pixel_waveform[min(int(arrival_time / float(current_params.t_sampling)), len(pixel_waveform)-1)]],
                        mode='markers',
                        marker=dict(
                            symbol='star',
                            size=marker_size,
                            color='blue',
                            line=dict(width=1, color='darkblue')
                        ),
                        name='Truth segments',
                        showlegend=False,
                        hovertemplate=f'Truth arrival: {arrival_time:.2f} µs<br>Energy: {energy:.3f} MeV<extra></extra>' if has_energy else f'Truth arrival: {arrival_time:.2f} µs<extra></extra>'
                    ),
                    row=1, col=1
                )
        else:
            print(f"   No truth segments found near pixel (within {pixel_tolerance:.2f} cm)")
        
        # Add the hit location in 3D
        # Show some context: nearby truth segments and all hits on this pixel
        nearby_distance = 50  # cm
        nearby_tracks = []
        for track in tracks_np:
            x_start = track[x_start_idx]
            y_start = track[y_start_idx]
            z_start = track[z_start_idx]
            x_end = track[x_end_idx]
            y_end = track[y_end_idx]
            z_end = track[z_end_idx]
            
            # Check if track is near the hit
            dist_start = np.sqrt((x_start - hit_x)**2 + (y_start - hit_y)**2 + (z_start - hit_z)**2)
            dist_end = np.sqrt((x_end - hit_x)**2 + (y_end - hit_y)**2 + (z_end - hit_z)**2)
            
            if dist_start < nearby_distance or dist_end < nearby_distance:
                nearby_tracks.append(track)
        
        # Add nearby truth segments
        if nearby_tracks:
            nearby_tracks = np.array(nearby_tracks)
            n_nearby = len(nearby_tracks)
            x_lines_nearby = np.empty(n_nearby * 3)
            y_lines_nearby = np.empty(n_nearby * 3)
            z_lines_nearby = np.empty(n_nearby * 3)
            
            x_lines_nearby[0::3] = nearby_tracks[:, x_start_idx]
            x_lines_nearby[1::3] = nearby_tracks[:, x_end_idx]
            x_lines_nearby[2::3] = np.nan
            
            y_lines_nearby[0::3] = nearby_tracks[:, y_start_idx]
            y_lines_nearby[1::3] = nearby_tracks[:, y_end_idx]
            y_lines_nearby[2::3] = np.nan
            
            z_lines_nearby[0::3] = nearby_tracks[:, z_start_idx]
            z_lines_nearby[1::3] = nearby_tracks[:, z_end_idx]
            z_lines_nearby[2::3] = np.nan
            
            fig.add_trace(
                go.Scatter3d(
                    x=x_lines_nearby,
                    y=y_lines_nearby,
                    z=z_lines_nearby,
                    mode='lines',
                    name='Nearby Truth',
                    line=dict(color='blue', width=3)
                ),
                row=1, col=2
            )
        
        # Add all hits on this pixel as voxels
        for h_idx in hits_on_pixel:
            h_x = target_x[h_idx]
            h_y = target_y[h_idx]
            h_z = target_z[h_idx]
            
            vertices, faces = create_voxel(
                h_x, h_y, h_z,
                pixel_pitch, pixel_pitch, time_interval_cm
            )
            
            # Highlight the current hit differently
            if h_idx == hit_index:
                color = 'red'
                opacity = 0.9
                show_in_legend = True
                label = f'Hit {hit_index}'
            else:
                color = 'pink'
                opacity = 0.4
                show_in_legend = False
                label = f'Other hits'
            
            fig.add_trace(
                go.Mesh3d(
                    x=vertices[:, 0],
                    y=vertices[:, 1],
                    z=vertices[:, 2],
                    i=faces[:, 0],
                    j=faces[:, 1],
                    k=faces[:, 2],
                    color=color,
                    opacity=opacity,
                    showlegend=show_in_legend,
                    name=label
                ),
                row=1, col=2
            )
        
        # Update layout
        fig.update_xaxes(title_text="Time (µs)", row=1, col=1)
        fig.update_yaxes(title_text="Current (mA)", row=1, col=1)
        
        fig.update_layout(
            scene=dict(
                xaxis_title='X (cm)',
                yaxis_title='Y (cm)',
                zaxis_title='Z (cm)',
                aspectmode='data'
            ),
            height=500,
            width=1400,
            showlegend=True
        )
        
        fig.show()
        
    else:
        print("\n⚠️  wfs not available in target_output")
        print("Available keys:", list(target_output.keys()))
        print("\nShowing hit location only:")
        
        # Just show the hit location
        fig = go.Figure()
        
        # Find all hits on this pixel
        hits_on_pixel = np.where(target_output['hit_pixels'] == pixel_id)[0]
        
        # Add all hits as voxels
        for h_idx in hits_on_pixel:
            h_x = target_x[h_idx]
            h_y = target_y[h_idx]
            h_z = target_z[h_idx]
            
            vertices, faces = create_voxel(
                h_x, h_y, h_z,
                pixel_pitch, pixel_pitch, time_interval_cm
            )
            
            # Highlight the current hit differently
            if h_idx == hit_index:
                color = 'red'
                opacity = 0.9
            else:
                color = 'pink'
                opacity = 0.4
            
            fig.add_trace(go.Mesh3d(
                x=vertices[:, 0],
                y=vertices[:, 1],
                z=vertices[:, 2],
                i=faces[:, 0],
                j=faces[:, 1],
                k=faces[:, 2],
                color=color,
                opacity=opacity,
                name=f'Hit {h_idx}'
            ))
        
        fig.update_layout(
            scene=dict(
                xaxis_title='X (cm)',
                yaxis_title='Y (cm)',
                zaxis_title='Z (cm)',
                aspectmode='data'
            ),
            height=600,
            width=800
        )
        
        fig.show()

# Example usage:
print("=" * 60)
print("To view a hit's waveform, use:")
print("  show_hit_waveform(hit_index)")
print(f"\nValid hit indices: 0 to {len(target_adcs)-1}")
print(f"Total hits: {len(target_adcs)}")
print(f"Valid hits (prob>0, ADC>0): {mask.sum()}")
print("=" * 60)

To view a hit's waveform, use:
  show_hit_waveform(hit_index)

Valid hit indices: 0 to 1858
Total hits: 1859
Valid hits (prob>0, ADC>0): 1859


In [21]:
# Example: Show waveform for a specific hit
# Change the number to view different hits

show_hit_waveform(1151)  # Shows hit #100

📊 Hit #1151 Information:
   Position: X=11.75, Y=46.78, Z=17.91 cm
   ADC Value: 89.6
   Hit Probability: 1.000
   Pixel ID: 269596
   Waveform Index: 376
   Number of hits on this pixel: 4
   Hit times (µs): [79.4, 166.0, 169.4, 171.60000000000002]


   Found 101 truth segments near pixel
   Detector: Anode at Z=0.16 cm, Cathode at Z=-30.59 cm
   Drift velocity: 0.1587 cm/µs
   Truth z positions (cm): ['2.37', '2.37', '2.38', '2.38', '2.39'] ...
   Drift distances (cm): ['2.21', '2.21', '2.22', '2.22', '2.23'] ...
   Truth arrival times (µs): ['13.91', '13.95', '13.98', '14.01', '14.04', '14.07', '14.11', '14.14', '14.17', '14.20']
